# 🐍 Python od podstaw — Moduł 8: Przetwarzanie tekstu — wyrażenia regularne i CSV

### Gdy `.split(",")` już nie wystarcza

W module 5 parsowałeś/aś plik CSV ręcznie, przez `.split(",")` — działało, ale łatwo
się psuło (np. przecinek wewnątrz wartości od razu wszystko rozjeżdża). Ten moduł to
dwa narzędzia do poważniejszej pracy z tekstem: wyrażenia regularne (`re`) do
wyszukiwania wzorców, i moduł `csv` do naprawdę poprawnej obsługi plików CSV.

## Spis treści

1. [Metody stringów — szybkie przypomnienie](#sec1)
2. [Wyrażenia regularne — podstawy](#sec2)
3. [Wzorce — klasy znaków i kwantyfikatory](#sec3)
4. [Grupy i `re.sub()`](#sec4)
5. [Moduł `csv` — dlaczego nie samo `.split(",")`](#sec5)
6. [`csv.DictReader` i `csv.DictWriter`](#sec6)
7. [Ciekawostka: kiedy NIE używać regexów](#sec7)
8. [Podsumowanie modułu](#sec8)
9. [Ćwiczenia](#sec9)

---

<a id="sec1"></a>
## 1. Metody stringów — szybkie przypomnienie

Zanim sięgniesz po regex, sprawdź, czy zwykłe metody stringów nie wystarczą — są
szybsze i czytelniejsze dla prostych przypadków.

In [ ]:
tekst = "  Python jest super!  "

print(tekst.strip())              # usuwa białe znaki z początku/końca
print(tekst.strip().lower())      # małe litery
print(tekst.strip().replace("super", "świetny"))
print(tekst.strip().split(" "))   # podział na listę słów
print("-".join(["a", "b", "c"]))  # sklejenie listy w string
print("Python" in tekst)          # sprawdzenie przynależności
print(tekst.strip().startswith("Python"))

<a id="sec2"></a>
## 2. Wyrażenia regularne — podstawy

**Wyrażenie regularne** (regex) to wzorzec opisujący, jak ma wyglądać szukany fragment
tekstu — dużo potężniejszy niż `.find()`, bo pozwala szukać np. „ciąg cyfr”, a nie
konkretnej wartości. W Pythonie obsługuje je moduł `re`.

In [ ]:
import re

tekst = "Zadzwoń pod numer 123-456-789 albo napisz na test@example.com"

# re.search() - szuka PIERWSZEGO dopasowania w dowolnym miejscu tekstu
dopasowanie = re.search(r"\d{3}-\d{3}-\d{3}", tekst)
print(dopasowanie.group())   # cały dopasowany fragment

# re.findall() - zwraca listę WSZYSTKICH dopasowań
maile = re.findall(r"\S+@\S+\.\S+", tekst)
print(maile)

# re.sub() - podmienia dopasowania na inny tekst
ocenzurowany = re.sub(r"\d{3}-\d{3}-\d{3}", "[UKRYTE]", tekst)
print(ocenzurowany)

> ⚠️ **Prefiks `r` przed wzorcem**
>
> Wzorce regex prawie zawsze zapisuje się jako *raw string* - z literą `r` przed cudzysłowem (`r"\d+"`). Bez tego Python próbowałby interpretować `\d` jako sekwencję ucieczki (jak `\n`), co psuje wzorzec. `r"..."` mówi Pythonowi: „nie przetwarzaj tu żadnych backslashy, przekaż je regexowi dosłownie”.

<a id="sec3"></a>
## 3. Wzorce — klasy znaków i kwantyfikatory

Najważniejsze elementy budowy wzorca:

| Symbol | Znaczenie |
|---|---|
| `\d` | dowolna cyfra (0-9) |
| `\w` | dowolny znak "słowny" (litera, cyfra, `_`) |
| `\s` | dowolny biały znak (spacja, tabulacja, nowa linia) |
| `.` | dowolny znak (poza nową linią) |
| `*` | poprzedni element: 0 lub więcej razy |
| `+` | poprzedni element: 1 lub więcej razy |
| `?` | poprzedni element: 0 lub 1 raz (opcjonalny) |
| `{n,m}` | poprzedni element: od `n` do `m` razy |
| `^` / `$` | początek / koniec tekstu (lub linii) |

In [ ]:
import re

teksty = ["kot123", "pies", "5 kotów", "ptak99xyz"]

for t in teksty:
    czy_pasuje = re.search(r"\d+", t) is not None   # czy zawiera choć jedną cyfrę
    print(f"{t}: {czy_pasuje}")

# Dopasowanie CAŁEGO stringu (^ i $) - tylko same cyfry, nic więcej:
for t in ["12345", "123a5", "abc"]:
    czy_sama_liczba = re.match(r"^\d+$", t) is not None
    print(f"{t}: {czy_sama_liczba}")

<a id="sec4"></a>
## 4. Grupy i `re.sub()`

Nawiasy `(...)` we wzorcu tworzą **grupę** — fragment dopasowania, do którego można się
później odwołać osobno, np. do wyciągnięcia tylko jednej części dopasowanego tekstu.

In [ ]:
import re

data_tekst = "Spotkanie jest 15-03-2026"
dopasowanie = re.search(r"(\d{2})-(\d{2})-(\d{4})", data_tekst)

print(dopasowanie.group())    # cały fragment: 15-03-2026
print(dopasowanie.group(1))   # pierwsza grupa: dzień -> 15
print(dopasowanie.group(2))   # druga grupa: miesiąc -> 03
print(dopasowanie.group(3))   # trzecia grupa: rok -> 2026

# re.sub() z grupami - zamiana formatu DD-MM-YYYY na YYYY/MM/DD:
nowy_format = re.sub(r"(\d{2})-(\d{2})-(\d{4})", r"\3/\2/\1", data_tekst)
print(nowy_format)

<a id="sec5"></a>
## 5. Moduł `csv` — dlaczego nie samo `.split(",")`

W module 5 dzieliłeś/aś linie CSV przez `.split(",")`. Problem: co, jeśli wartość
**sama zawiera przecinek** (np. `"Kowalski, Jan",25`)? Ręczny split się rozjedzie.
Wbudowany moduł `csv` poprawnie obsługuje cudzysłowy, przecinki wewnątrz wartości i
inne brzydkie przypadki brzegowe.

In [ ]:
import csv

# Stwórzmy plik CSV z nagłówkiem i przecinkiem WEWNĄTRZ wartości:
with open("osoby.csv", "w", newline="", encoding="utf-8") as plik:
    zapisujacy = csv.writer(plik)
    zapisujacy.writerow(["imie_nazwisko", "wiek", "miasto"])
    zapisujacy.writerow(["Jan Kowalski, Junior", 25, "Warszawa"])
    zapisujacy.writerow(["Ania Nowak", 30, "Kraków"])

# Odczyt - csv.reader poprawnie rozpozna, że przecinek w "Junior," jest częścią wartości:
with open("osoby.csv", "r", newline="", encoding="utf-8") as plik:
    czytajacy = csv.reader(plik)
    for wiersz in czytajacy:
        print(wiersz)

> ⚠️ **Parametr `newline=""`**
>
> Przy pracy z plikami CSV w Pythonie zawsze otwieraj plik z `newline=""` (pusty string) - bez tego, na Windowsie mogą pojawić się dodatkowe puste wiersze między każdym rekordem, bo system i moduł `csv` inaczej interpretują znaki końca linii.

<a id="sec6"></a>
## 6. `csv.DictReader` i `csv.DictWriter`

Zamiast list (gdzie trzeba pamiętać, że indeks `0` to imię, a `1` to wiek),
`DictReader`/`DictWriter` operują na słownikach z nazwami kolumn jako kluczami —
czytelniej i bezpieczniej.

In [ ]:
import csv

with open("osoby.csv", "r", newline="", encoding="utf-8") as plik:
    czytajacy = csv.DictReader(plik)
    for wiersz in czytajacy:
        print(wiersz)                    # to zwykły słownik
        print(wiersz["imie_nazwisko"])   # dostęp po nazwie kolumny, nie po indeksie

# Zapis ze słowników:
dane = [
    {"produkt": "Chleb", "cena": 4.5},
    {"produkt": "Mleko", "cena": 3.2},
]

with open("produkty.csv", "w", newline="", encoding="utf-8") as plik:
    zapisujacy = csv.DictWriter(plik, fieldnames=["produkt", "cena"])
    zapisujacy.writeheader()
    zapisujacy.writerows(dane)

with open("produkty.csv", "r", encoding="utf-8") as plik:
    print(plik.read())

<a id="sec7"></a>
## 7. Ciekawostka: kiedy NIE używać regexów

Regex jest potężny, ale łatwo go nadużyć.

> 💡 **Ciekawostka**
>
> W środowisku programistów krąży żart: „Niektórzy ludzie, gdy mają problem, myślą: OK, użyję wyrażeń regularnych. Teraz mają dwa problemy”. Regex bywa nieczytelny, trudny do debugowania i łatwo przeoczyć skrajny przypadek. Do prostych rzeczy (czy string zaczyna się od czegoś, podział po przecinku) lepiej użyć zwykłych metod stringów - regex zostaw na sytuacje, gdzie wzorzec faktycznie jest zmienny (np. „dowolna liczba cyfr”, „email w dowolnym formacie”).

<a id="sec8"></a>
## 8. Podsumowanie modułu

Po tym module powinno być jasne:

- kiedy zwykłe metody stringów wystarczą, a kiedy warto sięgnąć po regex,
- podstawowe symbole wzorców (`\d`, `\w`, `\s`, `*`, `+`, `?`, `{n,m}`),
- różnicę między `re.search()`, `re.match()`, `re.findall()` i `re.sub()`,
- jak działają grupy w regexie,
- dlaczego moduł `csv` jest bezpieczniejszy niż ręczny `.split(",")`,
- jak używać `csv.DictReader`/`csv.DictWriter` do pracy na słownikach zamiast list.

Kolejny moduł: **testowanie kodu** — jak sprawdzić, że Twoje funkcje faktycznie
działają poprawnie, w sposób automatyczny i powtarzalny, zamiast ręcznie sprawdzać
wzrokiem `print()`.

<a id="sec9"></a>
## 9. Ćwiczenia

Kilka zadań łączy regex z listami/pętlami, a druga część to praktyczna praca z CSV,
w tym poprawka podejścia z modułu 5.

> 📝 **Ćwiczenie 1: Czy to wygląda jak e-mail**
>
> Napisz funkcję `czy_email(tekst)`, sprawdzającą (uproszczonym regexem, nie musi być w 100% poprawna formalnie), czy string wygląda jak adres e-mail - coś, potem `@`, potem coś, kropka, coś. Przetestuj na kilku przykładach.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
import re

def czy_email(tekst):
    return re.match(r"^\S+@\S+\.\S+$", tekst) is not None

print(czy_email("kamil@example.com"))
print(czy_email("nie_email"))
print(czy_email("test@test"))
```
</details>

> 📝 **Ćwiczenie 2: Wyciąganie wszystkich liczb z tekstu**
>
> Mając string `tekst = "Zamówienie nr 42 zawiera 3 produkty za łącznie 129.99 zł"`, użyj `re.findall()`, żeby wyciągnąć wszystkie liczby (całkowite i zmiennoprzecinkowe) jako listę stringów.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
import re

tekst = "Zamówienie nr 42 zawiera 3 produkty za łącznie 129.99 zł"
liczby = re.findall(r"\d+\.?\d*", tekst)
print(liczby)
```
</details>

> 📝 **Ćwiczenie 3: Normalizacja białych znaków**
>
> Mając string z wieloma spacjami/tabulacjami pod rząd, np. `"To   jest    tekst  z\t\tbrzydkimi   odstępami"`, użyj `re.sub()`, żeby zamienić każdy ciąg białych znaków na pojedynczą spację.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
import re

tekst = "To   jest    tekst  z\t\tbrzydkimi   odstępami"
oczyszczony = re.sub(r"\s+", " ", tekst)
print(oczyszczony)
```
</details>

> 📝 **Ćwiczenie 4: Maskowanie numeru telefonu**
>
> Mając string `dane = "Kontakt: 600-100-200 lub 22-555-4321"`, użyj `re.sub()` z grupami, żeby zamaskować każdy numer, zostawiając tylko ostatnie 4 cyfry (np. `***-***-200`). Podpowiedź: przyjmij uproszczony wzorzec dla dwóch formatów osobno albo jeden bardziej ogólny.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
import re

dane = "Kontakt: 600-100-200 lub 22-555-4321"
zamaskowane = re.sub(r"\d{2,3}-\d{2,3}-(\d{3,4})", r"***-***-\1", dane)
print(zamaskowane)
```
</details>

> 📝 **Ćwiczenie 5: Poprawny odczyt CSV z przecinkiem w wartości**
>
> Stwórz plik `zamowienia.csv` (przez `csv.writer`) z kolumnami `klient,produkty,suma`, gdzie kolumna `produkty` dla jednego wiersza zawiera przecinek wewnątrz wartości (np. `"chleb, mleko, jajka"` jako JEDNA wartość - przekaż ją jako pojedynczy string w liście przekazanej do `.writerow()`). Wczytaj plik z powrotem przez `csv.reader` i pokaż, że ta kolumna wróciła jako jedna wartość, a nie trzy osobne.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
import csv

with open("zamowienia.csv", "w", newline="", encoding="utf-8") as plik:
    zapisujacy = csv.writer(plik)
    zapisujacy.writerow(["klient", "produkty", "suma"])
    zapisujacy.writerow(["Kamil", "chleb, mleko, jajka", 25.5])

with open("zamowienia.csv", "r", newline="", encoding="utf-8") as plik:
    czytajacy = csv.reader(plik)
    for wiersz in czytajacy:
        print(wiersz)   # "produkty" to nadal jedna wartość, mimo przecinków w środku
```
</details>

> 📝 **Ćwiczenie 6: `DictReader` w praktyce**
>
> Wczytaj plik `zamowienia.csv` z poprzedniego ćwiczenia przez `csv.DictReader` i dla każdego wiersza wypisz zdanie w stylu: „Kamil zamówił: chleb, mleko, jajka za 25.5 zł”.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
import csv

with open("zamowienia.csv", "r", newline="", encoding="utf-8") as plik:
    czytajacy = csv.DictReader(plik)
    for wiersz in czytajacy:
        print(f"{wiersz['klient']} zamówił: {wiersz['produkty']} za {wiersz['suma']} zł")
```
</details>

> 📝 **Ćwiczenie 7: Zapis wyników przez `DictWriter`**
>
> Mając listę słowników `wyniki = [{'gracz': 'Kamil', 'punkty': 120}, {'gracz': 'Ania', 'punkty': 95}]`, zapisz ją do pliku `wyniki.csv` przez `csv.DictWriter` (z nagłówkiem), a potem wczytaj i wypisz z powrotem, żeby potwierdzić poprawność.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
import csv

wyniki = [
    {"gracz": "Kamil", "punkty": 120},
    {"gracz": "Ania", "punkty": 95},
]

with open("wyniki.csv", "w", newline="", encoding="utf-8") as plik:
    zapisujacy = csv.DictWriter(plik, fieldnames=["gracz", "punkty"])
    zapisujacy.writeheader()
    zapisujacy.writerows(wyniki)

with open("wyniki.csv", "r", encoding="utf-8") as plik:
    print(plik.read())
```
</details>

> 🔥 **Ćwiczenie 8 (wyzwanie): Parser logów serwera**
>
> Stwórz plik `serwer.log` (kilka linii) w formacie: `2026-03-15 10:22:01 192.168.1.10 GET /index.html 200` (data, czas, IP, metoda, ścieżka, kod statusu). Napisz regex z grupami, który wyciągnie z każdej linii: adres IP, metodę HTTP i kod statusu. Policz i wypisz, ile żądań zakończyło się kodem błędu (status `>= 400`) - potraktuj status jako liczbę po wyciągnięciu z grupy.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
import re

with open("serwer.log", "w", encoding="utf-8") as plik:
    plik.write("2026-03-15 10:22:01 192.168.1.10 GET /index.html 200\n")
    plik.write("2026-03-15 10:22:05 192.168.1.11 POST /login 401\n")
    plik.write("2026-03-15 10:22:09 192.168.1.10 GET /missing.html 404\n")
    plik.write("2026-03-15 10:22:12 192.168.1.12 GET /index.html 200\n")

wzorzec = r"(\d{1,3}(?:\.\d{1,3}){3}) (GET|POST|PUT|DELETE) \S+ (\d{3})"
bledy = 0

with open("serwer.log", "r", encoding="utf-8") as plik:
    for linia in plik:
        dopasowanie = re.search(wzorzec, linia)
        if not dopasowanie:
            continue
        ip, metoda, status = dopasowanie.groups()
        status = int(status)
        print(f"IP: {ip}, metoda: {metoda}, status: {status}")
        if status >= 400:
            bledy += 1

print(f"Liczba żądań z błędem: {bledy}")
```

Podpowiedź: `(?:...)` to grupa «nie-przechwytująca» - działa jak zwykła grupa do powtarzania fragmentu wzorca (`\.\d{1,3}` powtórzone 3 razy dla adresu IP), ale nie zaśmieca listy wyników z `.groups()`.
</details>

---

### Co dalej?

Jeśli parser logów poszedł w miarę gładko, jesteś gotowy/a na **Moduł 9: testowanie
kodu** — jak automatycznie sprawdzać, że Twoje funkcje (włącznie z tymi napisanymi w
tym module) faktycznie robią to, co powinny.